# Train validation test split 80:10:10

In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split 

In [2]:
df = pd.read_parquet('../../data/processed_grouped_data.parquet')

In [3]:
df.head()

,amount,isFraud,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,orig_tx_count_same_step,orig_tx_cumcount,orig_prev_amount,...,dest_tx_cumcount,dest_prev_amount,dest_amount_ratio,dest_steps_since_last,pair_tx_cumcount,is_new_dest,pair_total_amount,hour_of_day,day,is_night
0,257348.03,0,1,0,0,0,0,1,1,0.0,...,4,598674.03,0.429863,0.0,0,1,257348.03,1,0,1
1,7512.61,0,0,0,0,1,0,1,1,0.0,...,1,0.00,0.000000,-1.0,0,1,7512.61,1,0,1
2,663.34,0,0,0,0,1,0,1,1,0.0,...,1,0.00,0.000000,-1.0,0,1,663.34,1,0,1
3,377287.80,0,1,0,0,0,0,1,1,0.0,...,1,0.00,0.000000,-1.0,0,1,377287.80,1,0,1
4,4252.82,0,0,0,0,1,0,1,1,0.0,...,1,0.00,0.000000,-1.0,0,1,4252.82,1,0,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 22 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   amount                   float64
 1   isFraud                  int64  
 2   type_CASH_IN             int64  
 3   type_CASH_OUT            int64  
 4   type_DEBIT               int64  
 5   type_PAYMENT             int64  
 6   type_TRANSFER            int64  
 7   orig_tx_count_same_step  int64  
 8   orig_tx_cumcount         int64  
 9   orig_prev_amount         float64
 10  orig_amount_ratio        float64
 11  steps_since_last_tx      float64
 12  dest_tx_cumcount         int64  
 13  dest_prev_amount         float64
 14  dest_amount_ratio        float64
 15  dest_steps_since_last    float64
 16  pair_tx_cumcount         int64  
 17  is_new_dest              int64  
 18  pair_total_amount        float64
 19  hour_of_day              int64  
 20  day                      int64  
 21  is_night

In [5]:
train_ratio = .8
val_ratio = .1
test_ratio = .1

In [6]:
train_idx = int(len(df) * train_ratio)
val_idx = int(len(df) * (train_ratio + val_ratio))

train_df = df.iloc[:train_idx]
val_df = df.iloc[train_idx:val_idx]
test_df = df.iloc[val_idx:]

In [7]:
# train test split
X_train = train_df.drop('isFraud', axis=1)
y_train = train_df['isFraud']

X_val = val_df.drop('isFraud', axis=1)
y_val = val_df['isFraud']

X_test = test_df.drop('isFraud', axis=1)
y_test = test_df['isFraud']

del df, train_df, val_df, test_df

import gc
gc.collect()

0

# Undersampling the majority data (non-fraud transactions) to avoid class imbalance

In [8]:
from imblearn.under_sampling import RandomUnderSampler

In [23]:
rus = RandomUnderSampler(sampling_strategy=.4, random_state=42)
X_train_resampled, y_train_resampled = rus.fit_resample(X_train, y_train)

In [24]:
print(f"{y_train_resampled.value_counts()}")

isFraud
0    9895
1    3958
Name: count, dtype: int64


# Save the train validation test data

In [21]:
import joblib

In [25]:
folder_path = '../../data/'
joblib.dump(X_train_resampled, folder_path + "x_train.joblib")
joblib.dump(y_train_resampled, folder_path + "y_train.joblib")
joblib.dump(X_val, folder_path + "x_val.joblib")
joblib.dump(y_val, folder_path + "y_val.joblib")
joblib.dump(X_test, folder_path + "x_test.joblib")
joblib.dump(y_test, folder_path + "y_test.joblib")

['../../data/y_test.joblib']